In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import argparse
import warnings
warnings.filterwarnings("ignore")

In [ ]:

class CodeBERTTrainer:
    def __init__(self, max_length=512, model_name="bert-base-uncased"):
        self.max_length = max_length
        self.model_name = model_name
        self.tokenizer = None
        self.model = None
        self.num_labels = None,
        # load dataset from Kaggle input directory when training on Kaggle
    
        # self.train_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_training_set_1.parquet"
        # self.val_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_validation_set.parquet"
        # self.test_path = "/kaggle/input/semeval-2026-task13/SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"
       
        # load dataset locally from HuggingFace hub
        self.train_path= 'hf://datasets/DaniilOr/SemEval-2026-Task13/task_a/task_a_training_set_1.parquet', 
        self.validation_path= 'hf://datasets/DaniilOr/SemEval-2026-Task13/task_a/task_a_validation_set.parquet', 
        self.test_path= 'hf://datasets/DaniilOr/SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet'

        
    def load_and_prepare_data(self):
        
        try:
            df = pd.read_parquet(self.train_path)
            
            print(f"Dataset columns: {df.columns.tolist()}")
            print(f"Sample data:\n{df.head()}")
            
            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")
            
            df = df.dropna(subset=['code', 'label'])
            
            df['label'] = df['label'].astype(int)
            self.num_labels = df['label'].nunique()
            
            print(f"Number of unique labels: {self.num_labels}")
            print(f"Label range: {df['label'].min()} to {df['label'].max()}")
            print(f"Label distribution:\n{df['label'].value_counts().sort_index()}")

            val_df = pd.read_parquet(self.val_path)
            
            print(f"Train samples: {len(df)}, Validation samples: {len(val_df)}")
            
            return df, val_df
            
        except Exception as e:
            print(f"Error loading dataset: {e}")
            raise
    
    def initialize_model_and_tokenizer(self):
        print(f"Initializing {self.model_name} model and tokenizer...")
        
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        self.model = AutoModelForSequenceClassification.from_pretrained(
            self.model_name,
            num_labels=self.num_labels,
            problem_type="single_label_classification"
        ).to('cuda')
        
        print(f"Model initialized with {self.num_labels} labels")
    
    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
    
    def prepare_datasets(self, train_df, val_df):
        print("Preparing datasets for training...")
        
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset = Dataset.from_pandas(val_df[['code', 'label']])
        
        train_dataset = train_dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=['code']
        )
        val_dataset = val_dataset.map(
            self.tokenize_function,
            batched=True,
            remove_columns=['code']
        )
        
        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset = val_dataset.rename_column('label', 'labels')
        
        return train_dataset, val_dataset
    
    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
        
        return {
            'accuracy': accuracy,
            'f1': f1,
            'precision': precision,
            'recall': recall
        }
    
    def train(self, train_dataset, val_dataset, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):
        print("Starting training...")
        print(self.model)
        print(self.model.device)
        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            # warmup_steps=500,
            weight_decay=0.01,
            logging_dir='./logs',
            logging_steps=5,
            evaluation_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="linear",
            save_total_limit=2,
            report_to=[],
        )
        
        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)
        
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
        )
        print(f"Start training")
        trainer.train()
        
        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        
        print(f"Training completed. Model saved to {output_dir}")
        
        return trainer
    
    def evaluate_model(self, trainer, val_dataset):
        print("Evaluating model...")
        
        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids
        
        print("Classification Report:")
        print(classification_report(y_true, y_pred))
        
        return predictions
    
    def run_full_pipeline(self, output_dir="./results", num_epochs=3, batch_size=16, learning_rate=2e-5):
        try:
            train_df, val_df = self.load_and_prepare_data()
            
            self.initialize_model_and_tokenizer()
            
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            
            trainer = self.train(
                train_dataset, val_dataset, 
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate
            )
            
            self.evaluate_model(trainer, val_dataset)
            
            print("Pipeline completed successfully!")
            return trainer
            
        except Exception as e:
            print(f"Error in pipeline: {e}")
            raise
    

In [16]:
trainer_obj = CodeBERTTrainer()


In [17]:
trainer_obj.load_and_prepare_data()

Error loading dataset: cannot construct a FileSource from ('hf://datasets/DaniilOr/SemEval-2026-Task13/task_a/task_a_training_set_1.parquet',)


TypeError: cannot construct a FileSource from ('hf://datasets/DaniilOr/SemEval-2026-Task13/task_a/task_a_training_set_1.parquet',)